In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha.dataset import AlphaDataset
from vnpy.alpha.dataset.processor import (
    process_drop_na, process_fill_na, process_cs_norm, process_robust_zscore_norm
)
from vnpy.trader.constant import Interval
from functools import partial
import polars as pl
from pathlib import Path
import gc
from datetime import datetime, timedelta
#导入因子定义模块
from vnpy.alpha.factor_define import (
    FACTOR_REGISTRY,
    FACTOR_NAMES,
    PARAMS_REGISTRY
)

In [2]:
# ============================================================================
# Cell 2: 路径配置和AlphaLab创建
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'
FACTOR_PATH = LAB_PATH / 'factor'
# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [3]:
# ============================================================================
# Cell 3: 时间配置
# ============================================================================
# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime.now()
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end
interval2 = Interval.DAILY

# 训练跨度
train_start = datetime(2018, 1, 1)
train_end = datetime(2023, 12, 31)

# 验证跨度
valid_start = datetime(2024, 1, 1)
valid_end = datetime(2024, 12, 31)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

# 滞后时间
lag_days = 0

#未来时间
future_days = 6 #这周一开盘买入到下周一开盘卖出

In [4]:
# ============================================================================
# Cell 4: 创建AlphaDataset
# ============================================================================
df = lab.load_bar_df(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=start,
    end=end,
    extended_days=0,
    adjust_type= "none"
)
with pl.Config(tbl_rows=317, tbl_cols=None, fmt_str_lengths=None):
    print(df)
daily_open = df.select(['datetime', 'vt_symbol', 'open']).sort(['vt_symbol', 'datetime'])

dataset = AlphaDataset(
    df=daily_open,
    train_period=(train_start, train_end),
    valid_period=(valid_start, valid_end),
    test_period=(test_start, test_end),
    process_type=''
)

print('数据集创建完成')
print(daily_open.shape)

shape: (1_051_119, 10)
┌────────────┬──────────┬──────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ datetime   ┆ open     ┆ high     ┆ low      ┆ … ┆ turnover   ┆ open_inte ┆ vwap      ┆ vt_symbol │
│ ---        ┆ ---      ┆ ---      ┆ ---      ┆   ┆ ---        ┆ rest      ┆ ---       ┆ ---       │
│ datetime[μ ┆ f64      ┆ f64      ┆ f64      ┆   ┆ f64        ┆ ---       ┆ f64       ┆ str       │
│ s]         ┆          ┆          ┆          ┆   ┆            ┆ f64       ┆           ┆           │
╞════════════╪══════════╪══════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2018-01-02 ┆ 2.534772 ┆ 2.578583 ┆ 2.528513 ┆ … ┆ 5.37373765 ┆ 0.0       ┆ 2.558228  ┆ 601818.SS │
│ 00:00:00   ┆          ┆          ┆          ┆   ┆ e8         ┆           ┆           ┆ E         │
│ 2018-01-03 ┆ 2.553548 ┆ 2.578583 ┆ 2.54729  ┆ … ┆ 5.02284165 ┆ 0.0       ┆ 2.567665  ┆ 601818.SS │
│ 00:00:00   ┆          ┆          ┆          ┆   ┆ e8         ┆    

In [4]:
factors = FACTOR_NAMES # 从 factor_define 导入的列表

In [10]:
factors = ["streverse_1m"]

In [6]:
lab.missing_ratio(factors, start = datetime(2026, 3, 1), end = datetime(2026, 3, 31))

开始加载...
late_skew_ret: 加载完成，共 300 只股票，1 个月数据
late_skew_ret: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
down_vol_perc: 加载完成，共 300 只股票，1 个月数据
down_vol_perc: 总行数 6,600, null数 8 (0.1212%), NaN数 0 (0.0000%)
开始加载...
corr_ret_lastret: 加载完成，共 300 只股票，1 个月数据
corr_ret_lastret: 总行数 6,600, null数 0 (0.0000%), NaN数 8 (0.1212%)
开始加载...
corr_close_nextopen: 加载完成，共 300 只股票，1 个月数据
corr_close_nextopen: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc2: 加载完成，共 300 只股票，1 个月数据
volume_perc2: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc3: 加载完成，共 300 只股票，1 个月数据
volume_perc3: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc4: 加载完成，共 300 只股票，1 个月数据
volume_perc4: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc5: 加载完成，共 300 只股票，1 个月数据
volume_perc5: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc6: 加载完成，共 300 只股票，1 个月数据
volume_perc6: 总行数 6,600, null数 0 (0.0000%), NaN数 0 (0.0000%)
开始加载...
volume_perc7: 加载完成，共 

In [16]:
# ============================================================================
# Cell 6:
# ============================================================================
def generate_lag_features_for_factor(factor_name, factor_df, dataset, lag_days=10):
    """
    为单个因子生成滞后特征，立即add_feature到dataset，然后释放内存

    Args:
        factor_name: str
        factor_df: 来自（load_specific_factor）
        dataset: AlphaDataset实例
        lag_days: 滞后天数
    """
    # 按股票分割成字典
    factor_dict = {}
    for symbol in factor_df['vt_symbol'].unique():
        symbol_df = factor_df.filter(pl.col('vt_symbol') == symbol)
        factor_dict[symbol] = symbol_df

    # 按滞后天数循环
    for lag in range(0, lag_days + 1):
        # 为股票生成lag特征
        lag_dfs = []

        for symbol, df in factor_dict.items():
            # 生成滞后
            lag_df = df.with_columns([
                pl.col('data').shift(lag).alias('data')
            ])
            lag_dfs.append(lag_df)

        # 合并所有股票的该lag特征
        if lag_dfs:
            merged_lag = pl.concat(lag_dfs).sort(['vt_symbol', 'datetime'])
            if lag == 0:
                feature_name = f'{factor_name}'
            else:
                feature_name = f'{factor_name}_lag_{lag}'

            # 添加到dataset
            dataset.add_feature(feature_name, result=merged_lag)

            # 释放内存
            del merged_lag, lag_dfs
            gc.collect()

    print(f'{factor_name}: 添加 {lag_days} 个滞后特征')

    # 释放原始因子数据
    del factor_dict
    gc.collect()

In [ ]:
 # ============================================================================
# Cell 7: 逐个因子处理：加载 → 生成滞后 → add_feature → 释放
# ============================================================================
factors = FACTOR_NAMES1 + FACTOR_NAMES2
print('开始流式处理因子...')
print(f'需处理 {len(factors)} 个因子，每个因子生成 {lag_days} 个滞后特征')
print(f'共 {len(factors) * lag_days} 个特征\n')

for i, factor_name in enumerate(factors, 1):
    print(f'\n[{i}/{len(factors)}] 处理因子: {factor_name}')

    # 1. 加载单个因子
    factor_df = load_specific_factor(factor_name, FACTOR_PATH, start, end, component_symbols)

    if  factor_df.is_empty():
        print(f'{factor_name} 无数据,无法滞后，pass')
        continue

    # 2. 生成滞后特征并立即添加到dataset，然后释放
    generate_lag_features_for_factor(factor_name, factor_df, dataset, lag_days=lag_days)

    # 3. 强制垃圾回收
    gc.collect()

print(f'\n✓ 所有因子处理完成！dataset共 {len(dataset.feature_results)} 个特征')

开始流式处理因子...
需处理 64 个因子，每个因子生成 0 个滞后特征
共 0 个特征


[1/64] 处理因子: late_skew_ret
开始加载...
late_skew_ret: 加载完成，共 553 只股票，101 个月数据
late_skew_ret: 添加 0 个滞后特征

[2/64] 处理因子: down_vol_perc
开始加载...
down_vol_perc: 加载完成，共 553 只股票，101 个月数据
down_vol_perc: 添加 0 个滞后特征

[3/64] 处理因子: corr_ret_lastret
开始加载...


In [10]:
# ============================================================================
# Cell 8: 设置标签：未来收益率
# ============================================================================
dataset.set_label(f'(ts_delay(open, -{future_days}) / ts_delay(open,-1)) - 1')
print(f'标签设置完成: 未来{future_days}日收益率(包含今日)')

标签设置完成: 未来2日收益率(包含今日)


In [11]:
# ============================================================================
# Cell 9: 准备数据（计算特征和标签，合并到result_df）
# ============================================================================
# 收集指数成分过滤器
filters: dict[str, list[str]] = lab.load_component_filters2(vt_index_symbol, start, end)
dataset.prepare_data(filters = filters)
print('数据准备完成')

2026-05-09 20:03:57 开始计算表达式因子特征


100%|██████████| 1/1 [00:02<00:00,  2.86s/it]

2026-05-09 20:04:00 开始合并结果数据因子特征



100%|██████████| 28/28 [00:01<00:00, 19.77it/s]


2026-05-09 20:04:02 开始筛选成分股数据


100%|██████████| 553/553 [00:01<00:00, 347.22it/s]


数据准备完成


In [12]:
print(dataset.learn_df)

shape: (604_800, 31)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ avg_trade ┆ big_order ┆ intraday_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ _size     ┆ _net      ┆ price_eff ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iciency   ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.16691   ┆ 0.253168  ┆ … ┆ 1.1906e7  ┆ 0.156992  ┆ 0.091623  ┆ -0.02986 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 2        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆     

In [13]:
# ============================================================================
# Cell 10: 预处理器
# ============================================================================

# # 添加数据预处理器
# dataset.add_processor('learn', partial(process_drop_na))
# dataset.add_processor("learn", partial(process_drop_na, names=["label"]))
# dataset.add_processor("learn", partial(process_cs_norm, names=["label"], method="zscore"))


# 删除训练数据缺失的行
dataset.add_processor('learn', partial(process_drop_na))
# 对训练集特征、标签进行截面z-score标准化
dataset.add_processor('learn', partial(process_cs_norm, names=[col for col in dataset.learn_df.columns if col not in ["datetime","vt_symbol"]], method='zscore'))

# 对推理集特征、标签进行截面z-score标准化
dataset.add_processor('infer', partial(process_cs_norm, names=[col for col in dataset.learn_df.columns if col not in ["datetime","vt_symbol"]], method='zscore'))
# dataset.add_processor('infer', partial(process_robust_zscore_norm, fit_start_time = train_start, fit_end_time = train_end))
# dataset.add_processor('infer', partial(process_cs_norm, names=["label"], method='zscore'))
#填充推理集数据
dataset.add_processor('infer', partial(process_fill_na, fill_value = 0))
print('预处理器添加完成')


预处理器添加完成


In [14]:
# ============================================================================
# Cell 11:
# ============================================================================
# 处理数据（应用预处理器）
dataset.process_data()

In [15]:
# ============================================================================
# Cell 12: 保存dataset
# ============================================================================

DATASET_NAME = 'v5'
lab.save_dataset(DATASET_NAME, dataset)

print(f'数据集已保存: {DATASET_NAME}')

数据集已保存: v5


In [16]:
print(dataset.learn_df)

shape: (601_000, 31)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ datetime  ┆ vt_symbol ┆ late_skew ┆ down_vol_ ┆ … ┆ avg_trade ┆ big_order ┆ intraday_ ┆ label    │
│ ---       ┆ ---       ┆ _ret      ┆ perc      ┆   ┆ _size     ┆ _net      ┆ price_eff ┆ ---      │
│ datetime[ ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ iciency   ┆ f64      │
│ μs]       ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆          │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2018-01-0 ┆ 000001.SZ ┆ 0.196827  ┆ -0.299025 ┆ … ┆ 2.476964  ┆ 0.078719  ┆ 0.412051  ┆ -2.06371 │
│ 2         ┆ SE        ┆           ┆           ┆   ┆           ┆           ┆           ┆ 4        │
│ 00:00:00  ┆           ┆           ┆           ┆   ┆           ┆     